# Random Guide-Selection Strategies

Compares random strategies for selecting guide sets.

`guided_search.py` writes one `results.parquet` per run (a run == one
strategy/config at a single guide-set size `k`; see `config.json`).
`helpers.load_runs` stacks the runs listed in `RUNS` into a single tidy long
DataFrame — one row per leg, tagged with its display `mode` (with ` (k=<k>)`
appended) and carrying `r_<metric>` = metric / that seed's unguided baseline.
Because each run is one `k`, the plots put the individual **seed/goal pairs** on
the x-axis (sorted by value) rather than sweeping `k`; overlay runs at different
`k`'s by listing them in `RUNS`. Every plot is an Altair spec over that frame
(see `plots.py`), so the cells below are a handful of declarative calls.

In [ ]:
import altair as alt
import polars as pl

import helpers as H
import plots as P

# Recessive grids, top legends, the validated categorical palette (see plots.THEME).
alt.theme.register("analysis", enable=True)(lambda: P.THEME)

# Datasets here comfortably exceed Altair's 5k-row default guard; we render inline.
alt.data_transformers.enable("default", max_rows=None)

# Guide-egraph column for each metric (sample.rs). The guide replay logs no
# per-iteration count, so `iters` has no guide column and is simply absent from
# the guide series.
GUIDE_COL = {
    "nodes": "guide_nodes",
    "classes": "guide_classes",
    "total_time": "guide_time",
    "memory": "guide_memory",
}


def absolute_long(df: pl.DataFrame, metrics: list[str]) -> pl.DataFrame:
    """Melt reached legs into tidy (metric, series, value) rows in native units.

    Emits three series per pair/metric: ``leg`` (the guided leg's own cost),
    ``baseline`` (that seed's unguided full-eqsat cost, `base_<metric>`), and
    ``guide`` (the per-seed guide replay, `guide_<metric>`; no `iters`). Keeping
    them in native units lets a plot show each phase's absolute cost — and its own
    resource limit — side by side. Keeps the pair keys so `abs_pair_strip` can
    collapse a pair's attempts and rank pairs along x.
    """
    d = df.filter(pl.col("reached"))

    def melt(frame: pl.DataFrame, cols: list[str], series: str) -> pl.DataFrame:
        return (
            frame.unpivot(
                index=["mode", "seed", "goal", "pair"],
                on=cols,
                variable_name="metric",
                value_name="value",
            )
            .with_columns(pl.lit(series).alias("series"))
        )

    leg = melt(d.select(["mode", "seed", "goal", "pair", *metrics]), metrics, "leg")
    baseline = melt(
        d.select(["mode", "seed", "goal", "pair", *[pl.col(f"base_{m}").alias(m) for m in metrics]]),
        metrics,
        "baseline",
    )
    guide_metrics = [m for m in metrics if m in GUIDE_COL]
    guide = melt(
        d.select(
            ["mode", "seed", "goal", "pair", *[pl.col(GUIDE_COL[m]).alias(m) for m in guide_metrics]]
        ),
        guide_metrics,
        "guide",
    )
    return pl.concat([leg, baseline, guide], how="vertical")

In [ ]:
# Configuration: runs to overlay as (display_name, run-dir pattern). Each run is
# one strategy/config at a single k; its k is folded into the mode label. To
# compare k's, list several single-k runs here — each becomes its own mode.
RUNS = [
    ("10k limit", "run.1"),
    # ("no-replacement k=10", "run.8"),
    # ("smallest-novel", "run.9"),
]

# Cost metrics carried through the plots. `nodes_per_class` is available in the
# frame too; add it to any plot's metric list to surface it. `memory` is jemalloc
# live-heap bytes (stats.allocated), folded as max(leg, guide) and ratioed against
# the seed baseline.
METRICS = ["iters", "nodes", "classes", "total_time", "memory"]

df, meta = H.load_runs(RUNS)
gr = H.goal_reach(df)

## Absolute cost vs. per-seed baseline

Each reached leg's cost in **native units**, with two references overlaid: the
seed's unguided full-eqsat `baseline` and the per-seed `guide` replay (blue =
leg, orange = baseline, green = guide). Metrics are columns and each run/mode is
its own row. Pairs are ranked by the leg value so the strip reads as a sorted leg
curve, with the baseline and guide points at the same x — the vertical gaps are
the guide's savings.

Dashed rules mark each series' resource limit, colored to match: the guide replay
**and** the legs run under the tighter `--stop-*` budget (here `--stop-nodes
10000`), while the baseline ran under the looser search-phase limit (100k nodes),
so on the nodes facet the leg/guide ceiling sits well below the baseline's.
`classes` is unbounded, so it has no line. The y-axis is log-scaled: absolute
costs span orders of magnitude across seeds.

In [ ]:
P.abs_pair_strip(
    absolute_long(df, METRICS),
    meta,
    title="Absolute cost vs per-seed baseline",
    y_title="cost (native units, log)",
    metrics=METRICS,
    limits=H.series_limit_frame(RUNS, METRICS),
)

## Reachability summary

Two panels: the per-pair reach rate (fraction of a pair's attempts that reached),
one point per seed/goal pair sorted within each mode; and a per-mode summary bar
for overall leg reach rate and goal coverage (goals with ≥1 success).

In [ ]:
P.reachability(df, gr, meta)

## Cost distribution by mode

Box plots of reached-leg cost, one facet per metric, a box per mode. Node count
includes the guide egraph (`max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
P.cost_boxplots(df, meta, ["iters", "nodes", "memory"])

## Summary table

In [ ]:
(
    df.group_by("mode")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("reached").mean().round(3).alias("reached_rate"),
        pl.col("reached").sum().alias("n_reached"),
        pl.len().alias("n_legs"),
    )
    .sort("mode")
)

## Goal-level reachability heatmap

Per-goal reach rate with goals on the y-axis (sorted by average reachability,
hardest at top) and one column per mode.

In [ ]:
P.reach_heatmap(gr, meta)

## Best absolute cost vs. baseline (per goal)

For each goal and series we take the **minimum** absolute cost over all attempts
that reached it, giving one point per goal per series, with goals sorted along x
by the best leg cost. The `baseline` and `guide` series sit at the same x, and
each series' dashed resource limit is drawn in its own color (leg/guide under the
tighter `--stop-*` budget, baseline under the search-phase limit). Native units,
log y; per-goal (not per-seed).

In [ ]:
BEST_METRICS = ["nodes", "total_time", "memory"]
P.abs_pair_strip(
    absolute_long(df, BEST_METRICS),
    meta,
    title="Best absolute cost vs baseline (per goal)",
    y_title="cost (native units, log)",
    metrics=BEST_METRICS,
    limits=H.series_limit_frame(RUNS, BEST_METRICS),
    group_by=["goal"],  # best (min cost) per goal, per series
    group_reduce="min",
)